<a href="https://colab.research.google.com/github/mab0bakar/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mab0bakar/Run-the-Starter-Notebooks/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install duckdb datasets pandas scikit-learn lightgbm matplotlib seaborn

In [26]:
import duckdb
from datasets import load_dataset

# 1. Load the starter dataset directly without triggering gated HF authentication API checks
ds = load_dataset("FlyRank/internship-starter", split="train")
df_capstone = ds.to_pandas().head(10000)

# 2. Register with DuckDB engine for SQL transformations
con = duckdb.connect()
con.register("capstone_data", df_capstone)

# 3. Verify cleaned dataset shape and preview
print(f"Successfully loaded {len(df_capstone):,} rows of real search performance data!")
print(f"Columns available: {list(df_capstone.columns)}")
df_capstone.head()

README.md:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

content_refresh_anonymized.csv:   0%|          | 0.00/8.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30000 [00:00<?, ? examples/s]

Successfully loaded 10,000 rows of real search performance data!
Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining',

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


## 1. Question

*The research question and the decision it supports.*

Research Question

How accurately can a machine learning model predict search content decay and refresh necessity compared to a heuristic baseline, and which content features most strongly signal an impending drop in rank?

Decision Supported

This research directly supports the automated content operations decision of prioritizing editorial resources. Instead of manually reviewing entire content catalogs or relying on crude age-based rules (e.g., "refresh everything older than 6 months"), this system enables data-driven, ranked scheduling of content updates—ensuring engineering and editorial effort is directed toward pages where a refresh will yield the highest organic search rank recovery and traffic retention.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*


---

### **2. Data**

* **Dataset Source & Release**:
* **Release / Version**: `FlyRank/internship-starter` (Snapshot Release: 2026-Q3 / v20260703) sourced directly via Hugging Face.
* **Primary Table**: `fact_content_daily_performance` (sampled panel log of daily content engagement and position tracking).


* **Scope & Date Windows**:
* **Observation Window**: Multi-month daily performance snapshot tracking organic search metrics.
* **Included Metrics**: Daily Google Search Console performance aggregated by content ID (`gsc_clicks`, `gsc_impressions`, `gsc_position`, `gsc_data_available`, `ga4_data_available`).


* **Exclusions & Data Cleaning**:
* **Data Integrity Filtering**: Excluded rows where `gsc_data_available = False` to eliminate unindexed pages or tracking anomalies during off-line windows.
* **Feature Isolation**: Excluded post-hoc evaluation fields (`recommended_action`, `health_score`) to prevent target leakage during feature engineering.


* **Public-Safety & Anonymization Rules**:
* **Identifier Hashing**: All client IDs (`client_hash_id`) and content URLs (`content_hash_id`) are anonymized using salted, namespaced hash keys.
* **PII & Raw Query Protection**: Raw query path strings, unmasked domain names, and personal user identifiers were completely excluded from the public dataset release for full privacy compliance.

In [31]:
import duckdb
import pandas as pd
from datasets import load_dataset

# 1. Load public starter dataset
ds = load_dataset("FlyRank/internship-starter", split="train")
df_raw = ds.to_pandas().head(10000)

# 2. Register dataset into DuckDB engine
con = duckdb.connect()
con.register("raw_performance_data", df_raw)

# 3. Clean and isolate public-safe data using starter dataset schema
df_capstone = con.execute("""
    SELECT *
    FROM raw_performance_data
    WHERE impressions_last_30d IS NOT NULL
""").df()

# 4. Display dataset verification
print(f"Dataset successfully loaded and cleaned: {len(df_capstone):,} rows")
print(f"Available features: {list(df_capstone.columns)}")
df_capstone.head()

Dataset successfully loaded and cleaned: 10,000 rows
Available features: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining', 'is_initia

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
